# Sprint 3 Live — Streaming, safe logs and evaluation

Use one personal Colab copy for four 60-minute concept workshops. Explain an idea, predict a result, run the supplied example, then interpret it. The example's names and paths are incidental; no Campus project, ZIP or business-case preparation is required. FieldCare project application belongs to Sprint 4.

## Learning outcomes and navigation

| Session | Exact learning outcome | Notebook section |
|---|---|---|
| LS09 | Enable streaming on the project service and observe the difference from the client side. | [Streaming and client state](#ls09) |
| LS10 | Add structured logging that captures request ID, model, tokens, latency, and errors — with redaction for personal data and sensitive prompts. | [Useful safe records](#ls10) |
| LS11 | Connect service logs back into the evaluation framework from Module A — no new eval framework built. | [A signal to an evaluation question](#ls11) |
| LS12 | Demo the streaming service with structured logging, redaction, and the eval feedback hook. | [Concept clinic](#ls12) |

Start with [setup](#setup-guide). Save the notebook and [download your Live ZIP](#save-work) after each session. Cell labels such as `code-05` are stable IDs used by the instructor guide; search the unique nearby heading or use Colab's table of contents.

`PROJECT` is the separate Live practice folder, not a Campus project. Edit only the files under the printed `PROJECT` path. `REPO` is a fresh source checkout. The supplied `live_*.py` composition files make route registration and middleware attachment visible; each service runs in its own owned child process. A Colab `127.0.0.1` address belongs to that runtime, not your laptop. No public tunnel or foreground Uvicorn call is needed.

### Keep your own reasoning

Add a text cell below each application and keep these entries blank until you try it:

| Prediction | Cell or edited source | Actual result | Explanation | Still untested |
|---|---|---|---|---|
| | | | | |

Provider-dependent streaming is optional to execute when access is available, but remains pending evidence for the streaming outcome when unavailable. A written event sequence or a deterministic clarification is not a substitute for an observed provider stream.


<a id="setup-guide"></a>
## 1. Set up or resume your Live workspace

1. **Save a personal notebook copy.** Run `setup` once. It clones current `main`, installs the reviewed lock and creates a separate marked Live folder. Expected: the folder, source revision and two approved model choices print; no provider call occurs.
2. **Inspect the Files panel.** Open the printed `PROJECT/service/fieldcare/` folder. Double-click a `.py` file to edit it, save with Ctrl/Cmd+S, then rerun the source-printing cell to verify the saved text. A source edit affects a child service only after its start cell runs again.
3. **Resume existing work carefully.** Rerunning setup preserves a marked Live folder, stops our previous service and generates new temporary caller keys. Rerun the relevant start cell and requests afterward. It clears current observation variables so a new run cannot borrow old results, while retaining edited source and saved evidence files. Earlier runtime logs remain on disk; match only fresh request IDs. If the folder exists without the Live marker, choose a new `LIVE_FOLDER`; do not delete unrelated work.

### Continue in a fresh runtime

Upload your own exported Live ZIP through Colab's Files panel (or provide its local path in Jupyter). Set `LIVE_ARCHIVE`, choose a `LIVE_FOLDER` that does not exist, and fill both `RESTORED_MODELS` from the saved model choices printed at export. Setup validates and restores source and selected evidence without executing it. Review restored `.py` files before starting them. Clear `LIVE_ARCHIVE` after successful restoration before rerunning setup in that same runtime.

Credentials, indexes, `var/requests.jsonl` and Python variables are not restored. Re-enter a provider key and prepare the index only for an intended provider run. Selected safe observations saved under `evidence/` are historical; make fresh requests to test a restored service. A Campus checkpoint is not a Live ZIP.

**Check:** locate `live_stream.py` and `live_logging.py`; keep `PREPARE_PROVIDER` and `RUN_PROVIDER` false until you explicitly choose provider work. If install/import fails, preserve your ZIP and notebook, reconnect a fresh runtime and rerun setup. Do not combine imported helpers from different source revisions in one kernel.


In [ ]:
from pathlib import Path
import os, sys, subprocess, tempfile, json, secrets, time
if globals().get("service") is not None:
    service.stop()
service = None
REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path(tempfile.mkdtemp(prefix="module-b-live-source-")) / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
import httpx
previous_helper_path = globals().get("LIVE_HELPER_PATH")
import importlib
if previous_helper_path in sys.path:
    sys.path.remove(previous_helper_path)
LIVE_HELPER_PATH = str(REPO / "src")
sys.path.insert(0, LIVE_HELPER_PATH)
# Select the new checkout even when setup is rerun in the same runtime.
for module_name in tuple(sys.modules):
    if module_name == "module_b" or module_name.startswith("module_b."):
        del sys.modules[module_name]
importlib.invalidate_caches()
from module_b.live import prepare_live_workspace, export_live_workspace, restore_live_workspace
from module_b.runtime import ServiceProcess
# On a fresh runtime, upload your own Live ZIP using the Files panel.
LIVE_ARCHIVE = ""  # Example: "/content/sprint-3-live-123.zip"; keep empty for new practice.
LIVE_FOLDER = Path.cwd() / "fieldcare-sprint-3-live"
# When restoring, explicitly select the same approved models used in that work.
RESTORED_MODELS = {"OPENROUTER_MODEL": "", "OPENROUTER_MODEL_V2": ""}
from module_b.openrouter import CHAT_MODELS, DEFAULT_CHAT_MODEL
if LIVE_ARCHIVE:
    if any(model not in CHAT_MODELS for model in RESTORED_MODELS.values()):
        raise ValueError("Fill both RESTORED_MODELS with your saved approved model choices before restoring.")
    PROJECT = restore_live_workspace(LIVE_ARCHIVE, LIVE_FOLDER)
else:
    PROJECT = prepare_live_workspace(REPO, LIVE_FOLDER)
CALLERS = {"dispatch": "FIELDCARE_DISPATCH_KEY", "partner": "FIELDCARE_PARTNER_KEY"}
caller_keys = {name: secrets.token_urlsafe(32) for name in CALLERS}
previous_models = {name: globals().get("service_env", {}).get(name) for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}
service_env = {variable: caller_keys[name] for name, variable in CALLERS.items()}
service_env["FIELDCARE_WORK_DIR"] = str((PROJECT / "var").resolve())
for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2"):
    service_env[name] = (RESTORED_MODELS[name] if LIVE_ARCHIVE else
                         previous_models[name] or os.environ.get(name, DEFAULT_CHAT_MODEL))
    if service_env[name] not in CHAT_MODELS:
        raise ValueError(f"Select an approved {name}.")
print("Configured models:", {name: service_env[name] for name in RESTORED_MODELS})
supported = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
incomplete = {"question": "Which filter checks are documented?"}
print("Live workspace:", PROJECT)
SOURCE_REVISION = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
# A new setup is a new observation context; saved evidence files remain intact.
for observation_name in (
    "RUN_PROVIDER", "PREPARE_PROVIDER", "stream_request_id", "buffered_request_id",
    "rejected_request_id", "clarification_request_id", "request_ids", "records",
    "selected_fields", "events", "terminal", "first_delta_seconds", "stream_elapsed_seconds",
    "buffered_elapsed_seconds", "stream_http_status", "buffered_http_status",
    "stream_transport_error", "buffered_transport_error", "report", "candidate", "minimized",
):
    globals().pop(observation_name, None)
print("Source main:", SOURCE_REVISION)
print("No provider or index is needed for health, validation and clarification.")

from module_b.live import start_live_service


<a id="ls09"></a>
## LS09 — Streaming and client state

**Outcome:** Enable streaming on the project service and observe the difference from the client side.

A buffered response returns a collected JSON object. A streamed response lets an incremental client consume events before the answer is complete. Here the service exposes NDJSON: one JSON event per line. The provider's internal SSE format is a different boundary. Retrieval happens before the first streamed answer event; streaming does not promise faster generation.

### Enable the route, then predict the boundary

1. Run `ls09-source`. In `stream_routes.py`, locate the ordinary JSON clarification branch, the async generator, each `yield encode_event(...)` and `StreamingResponse`.
2. Open `live_stream.py` in the Files editor. Enable the supplied router with `app.include_router(stream_router)` **before** `protected_post_paths(app)` and guard attachment. The starter already includes this line: retain exactly one registration. Explain why collecting protected paths before registration would miss the route. Save any edit and rerun the source cell.
3. Run `code-04` to start that composition. Expected: an owned loopback URL. Run `ls09-boundaries` once and compare your predictions with the status and content type. Missing key gives `401`; admitted invalid input gives `422`; missing context gives `200` with ordinary JSON clarification. None of these generates an answer.

The supplied policy allows each caller two admitted attempts per 60 seconds across protected routes. Invalid admitted input counts. Repeating requests can return `429`; wait for the current window or deliberately begin a new experiment by restarting. A restart is not evidence of quota renewal.

**Check before continuing:** why can a route named “stream” return JSON, and why does HTTP `200` alone not establish answer completion?


In [ ]:
print((PROJECT / "service/fieldcare/stream_routes.py").read_text())
print((PROJECT / "service/fieldcare/live_stream.py").read_text())

In [ ]:
service = start_live_service(PROJECT, "live_stream", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    for label, body, headers in [
        ("missing key", incomplete, {}),
        ("invalid", {}, {"X-API-Key": caller_keys["dispatch"]}),
        ("clarification", incomplete, {"X-API-Key": caller_keys["dispatch"]}),
    ]:
        response = client.post("/v1/diagnose-stream", json=body, headers=headers)
        print(label, response.status_code, response.headers.get("content-type"), response.json())

### Optional provider preparation — explicit opt-in

A supported generated-answer comparison needs provider access and an index. Leave `PREPARE_PROVIDER = False` for the provider-free path. To opt in, set it true, run the next cell and enter the private key through Colab Secrets or the hidden prompt. The cell stops our service, then invokes the supplied index-preparation command; embedding the synthetic documents uses provider quota.

Expected: preparation completes successfully. A healthy process alone proves neither provider access nor answer quality. If preparation fails, read the safe error, correct access/model configuration and retry deliberately; keep streaming observations pending. Do not paste a key into source, outputs or evidence. Start `ls09-observed-start` after successful preparation, then explicitly enable `RUN_PROVIDER` in `code-05`. Set both switches back to false after the intended run.


In [ ]:
PREPARE_PROVIDER = False
if PREPARE_PROVIDER:
    if service is not None:
        service.stop()
    from module_b.openrouter import require_openrouter_key
    require_openrouter_key(prompt=True)
    preparation_env = {**os.environ, **service_env,
                       "PYTHONPATH": str(PROJECT / "service") + os.pathsep + str(REPO / "src")}
    subprocess.run([sys.executable, "-m", "fieldcare.prepare_index"],
                   cwd=PROJECT, env=preparation_env, check=True)
else:
    print("Provider preparation pending; deterministic checks remain available.")

### Correlate an event trace with a request record

Run `ls09-observe-source` and read `live_observed.py`. Observation middleware is added last, outside the guard, so it can record rejected attempts too. Run `ls09-observed-start`; this starts the observed composition with the same enabled stream route and a new allowance window. Make fresh requests: observation cannot add records to earlier requests retroactively.

In `code-05`, locate `client.stream`, `iter_lines` and the first nonempty delta timer before running. With `RUN_PROVIDER = False`, expect an explicit pending message and no generated output. With prepared resources and the switch true, expect either actual events or an honest failure. A clarification remains JSON. `metadata` means ready, `delta` means partial text, `complete` confirms the stream's terminal completion, and `error` or transport loss leaves a failed or incomplete response. Optional `usage` may be absent.

Use two caller allowances for the streamed and buffered requests. Compare first text and reader finish with the buffered response time; these are separate calls, not a benchmark. The first nonempty delta measures arrival, not usefulness or quality.


In [ ]:
print((PROJECT / "service/fieldcare/live_observed.py").read_text())

In [ ]:
service = start_live_service(PROJECT, "live_observed", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
RUN_PROVIDER = False
stream_request_id = None
buffered_request_id = None
events = []
first_delta_seconds = None
stream_elapsed_seconds = None
buffered_elapsed_seconds = None
stream_http_status = None
buffered_http_status = None
stream_transport_error = None
buffered_transport_error = None
terminal = None
if RUN_PROVIDER:
    started = time.perf_counter()
    with httpx.Client(base_url=service.base_url, timeout=120) as client:
        try:
            with client.stream("POST", "/v1/diagnose-stream", json=supported,
                               headers={"X-API-Key": caller_keys["dispatch"]}) as response:
                stream_request_id = response.headers.get("x-request-id")
                stream_http_status = response.status_code
                print("Stream HTTP", stream_http_status, "request", stream_request_id)
                if response.status_code == 200 and "ndjson" in response.headers.get("content-type", ""):
                    for line in response.iter_lines():
                        if line:
                            event = json.loads(line)
                            events.append(event)
                            if event.get("type") == "delta" and event.get("text") and first_delta_seconds is None:
                                first_delta_seconds = time.perf_counter() - started
                            print(round(time.perf_counter()-started, 3), event)
                else:
                    response.read()
                    print("Non-stream result:", response.text)
        except (httpx.HTTPError, ValueError) as error:
            stream_transport_error = type(error).__name__
            print("Stream read/parse failure:", stream_transport_error, "; retain any partial events.")
        stream_elapsed_seconds = time.perf_counter() - started
        buffered_started = time.perf_counter()
        try:
            response = client.post("/v1/diagnose", json=supported,
                                   headers={"X-API-Key": caller_keys["partner"]})
            buffered_http_status = response.status_code
            buffered_request_id = response.headers.get("x-request-id")
            buffered_body = response.json() if "json" in response.headers.get("content-type", "") else response.text
            print("Buffered", buffered_http_status, buffered_request_id, buffered_body)
        except (httpx.HTTPError, ValueError) as error:
            buffered_transport_error = type(error).__name__
            print("Buffered read/parse failure:", buffered_transport_error)
        buffered_elapsed_seconds = time.perf_counter() - buffered_started
    terminal = next((e for e in reversed(events) if e.get("type") in {"complete", "error"}), None)
    print("Stream first text / reader finished (seconds):", first_delta_seconds, stream_elapsed_seconds)
    print("Buffered reader finished (seconds):", buffered_elapsed_seconds)
    print("Separate requests; these timings are observations, not a benchmark.")
    print("Terminal:", terminal, "Complete:", bool(terminal and terminal["type"] == "complete" and stream_transport_error is None))
    print("Usage:", next((e.get("tokens") for e in events if e.get("type") == "usage"), None))
else:
    print("Real stream and buffered AI comparison pending. No output is substituted.")


### Independent comparison — 11 minutes, then explain

Choose a consumer and predict whether it needs progressive text or a complete JSON result. With provider access, repeat the same supported input through the two clients in `code-05`, with allowance available. Record the actual IDs, first-text timing, terminal event and buffered outcome. Explain your consumer choice; do not infer better quality or a speed guarantee from two calls. Without access, inspect the two clients and keep the actual comparison pending.

For **written calibration only**, reason about `metadata → delta → error` and `metadata → delta → connection closed`. Neither supports a completed-answer claim. These are not observed runs. A transport failure can occur after partial text; inspect the printed transport status and any received events rather than treating HTTP `200` as completion. No forced provider interruption is promised.

**Peer check:** name which observation supports partial text, which supports completion, and what remains unknown. Save your own explanation in a text cell.


<a id="ls10"></a>
## LS10 — Useful safe records

**Outcome:** Add structured logging that captures request ID, model, tokens, latency, and errors — with redaction for personal data and sensitive prompts.

A structured log stores named fields so requests can be compared without copying their private contents. The request ID is a unique identifier for one invocation; it is different from an evaluation case ID.

| Field | What it can tell you | Interpretation limit |
|---|---|---|
| `request_id`, `route` | Which invocation and approved route | Join by exact ID, not by nearby time |
| `status_code`, `outcome`, `error_category` | HTTP status and terminal handling | A `200` stream can still fail later |
| `configured_model`, `model` | Selected model and approved provider-reported identity | Different claims; early returns make no model call |
| `tokens` | Reported usage when available | `None` means unknown, not zero |
| `latency_ms` | Observation middleware's server handling duration | Not the client's full elapsed time |
| Optional `first_content_ms` | Route timer to its first delta after retrieval | Not the client's pre-request-to-first-text timer |

Explain these meanings first. Then configure observation, make fresh deterministic requests and find exactly their terminal rows. Use LS09 provider IDs only when they came from an actual run and were recorded by the observed composition.


### Attach your own logging configuration
Run `ls10-configure-source`, then open the printed `PROJECT/service/fieldcare/live_logging.py` path in Colab’s Files editor. It imports the already secured streaming service but does not attach observation. Keep its required `LOG_FIELDS`; decide whether to add `"first_content_ms"` and explain which question it answers. Do not add prompts, headers or raw exceptions.

Add this supplied attachment once at the bottom of that file and save it. If restored source already contains it, retain one attachment rather than duplicating it:

```python
app.add_middleware(
    ObservationMiddleware,
    path=work_dir() / "requests.jsonl",
    routes=protected_post_paths(app),
    fields=LOG_FIELDS,
    approved_models=(openrouter_model(), openrouter_model_v2()),
)
```

This wraps the existing guard, so rejections are observed too. `configured_model` records the selected model when generation runs; `model` records an approved provider-reported identity, independently of optional token usage. These are different claims. Early rejections and clarifications have no model call.

Run `ls10-configure-source` again to confirm the saved edit, then `ls10-configure-start` to load it in a fresh process. Run `ls10-requests`, then `code-07`. Expected: a `401` rejection and a `200` clarification each have their own response UUID and exactly one matching JSONL row with your selected fields. The clarification finishes without a model call. The check waits briefly for the terminal row to be written.

If the ID is missing, inspect the attachment and restart after saving; an old log file does not prove the new composition is observed. If no current row matches, verify the printed workspace and fresh IDs instead of borrowing an old row. Repeated requests can use the allowance; check `429` before explaining an unexpected result.


In [ ]:
print(PROJECT / "service/fieldcare/live_logging.py")
print((PROJECT / "service/fieldcare/live_logging.py").read_text())


In [ ]:
# Start your edited composition; a fresh process loads the saved field selection.
service = start_live_service(PROJECT, "live_logging", service_env, service)
print("Own logging composition:", service.base_url)


In [ ]:
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    response = client.post("/v1/diagnose", json=incomplete)
    rejected_request_id = response.headers.get("x-request-id")
    print("Rejected", response.status_code, rejected_request_id)
    response = client.post("/v1/diagnose", json=incomplete,
                           headers={"X-API-Key": caller_keys["partner"]})
    clarification_request_id = response.headers.get("x-request-id")
    print("Admitted attempt", response.status_code, clarification_request_id)

In [ ]:
from module_b.observability import FIELDS
import ast
LOG_PATH = PROJECT / "var" / "requests.jsonl"
assert LOG_PATH.exists(), "Attach observation in live_logging.py, save and restart."
# Read the literal field tuple without importing or starting another service.
configuration = ast.parse((PROJECT / "service/fieldcare/live_logging.py").read_text())
selected_fields = ast.literal_eval(next(
    node.value for node in configuration.body if isinstance(node, ast.Assign)
    and any(isinstance(target, ast.Name) and target.id == "LOG_FIELDS" for target in node.targets)
))
request_ids = [rejected_request_id, clarification_request_id]
assert set(selected_fields) <= set(FIELDS), "Choose fields supported by the supplied safe schema."
# A response can arrive just before middleware appends its terminal row.
deadline = time.monotonic() + 2
while True:
    records = [json.loads(line) for line in LOG_PATH.read_text().splitlines() if line]
    if all(any(row.get("request_id") == rid for row in records) for rid in request_ids):
        break
    if time.monotonic() >= deadline:
        break
    time.sleep(0.05)
for request_id in request_ids:
    assert request_id, "This response has no observation ID; inspect the attachment."
    matches = [row for row in records if row["request_id"] == request_id]
    assert len(matches) == 1, "Find exactly this fresh request, not an older LS09 row."
    assert set(matches[0]) == set(selected_fields), "Saved fields differ from your configuration."
    print("Fresh saved record:", matches[0])
print("Configured fields verified:", selected_fields)
# If provider preparation is complete, rerun code-05 on this edited service.
# Correlate its new IDs separately; unavailable provider evidence stays pending.


### Minimize before saving, then check usefulness

An allowlist names the fields permitted to leave the request-handling context. `safe_record` also validates their values: approved routes/models, bounded categories, numeric measurements and a valid UUID. This is minimization for a fixed schema, not a universal personal-data detector or a filter applied only to displayed output.

Run `code-09`. Its synthetic marker goes only into a local candidate dictionary, never to the HTTP service or provider. Expected: the marker is absent from the minimized result while the request ID and outcome remain. The original candidate still exists in memory. Separately inspect the actual JSONL rows from `code-07`; neither this local marker example nor one saved file proves all logging destinations are safe.

### Independent field choice and marker check — 11 minutes

Choose a debugging question and explain whether `first_content_ms` helps answer it. Keep the required fields; save your configuration, restart if changed, and make fresh requests before rerunning `code-07`. Change the local synthetic marker in `code-09`, predict what is excluded and which useful values survive, then run it. Do not use real personal data or credentials.

Record one actual row's identity, model/usage/timing/error meaning, one retained useful value, and the limit of your minimization claim. **Peer check:** could an empty file also pass a marker search, and what additional observation establishes useful logging?


In [ ]:
from module_b.observability import safe_record
marker = "SYNTHETIC-NOT-A-CREDENTIAL"
observed = next(row for row in records if row["request_id"] == rejected_request_id)
candidate = {**observed, "prompt": marker, "headers": {"X-API-Key": marker},
             "_approved_models": (service_env["OPENROUTER_MODEL"],)}
minimized = safe_record(candidate, fields=selected_fields, routes=("/v1/diagnose", "/v1/diagnose-stream"))
assert marker not in json.dumps(minimized)
assert minimized["request_id"] == observed["request_id"]
assert minimized["outcome"] == observed["outcome"]
print("Local minimization result:", minimized)
print("Actual saved log has prompt/header fields:", any("prompt" in r or "headers" in r for r in records))

<a id="ls11"></a>
## LS11 — A signal to an evaluation question

**Outcome:** Connect service logs back into the evaluation framework from Module A — no new eval framework built.

A log describes request handling. A quality hypothesis asks whether behavior met a requirement. For example, a clarification observation can motivate “Does missing required context cause a request for details before guidance?” Latency or a `completed` record cannot answer that question.

1. Choose a safe observation from your Colab run and write a hypothesis before running evaluation. Keep its runtime request ID separate from an `eval_id`.
2. Run `code-11` to read the supplied original Module A cases and criteria. The guided pair demonstrates missing-context behavior and relevant-evidence selection. The concrete case names belong to the supplied example, not a project task.
3. Read the actual evaluator rows against those printed requirements and retain the design hash. Expected: results for the selected original cases, with the original document/tool/response-flag criteria unchanged. Accept an honest pass; no forced defect is required.

The evaluator reads `service/data/pipeline_design.json`, a deterministic reference design. The live OpenRouter graph does not use that design. A shared case or source revision does not make the evaluator a scorer of live generated wording or streaming. Use the same original framework; do not create a replacement score.

**Check:** state which criterion answers the hypothesis and one additional observation needed before claiming live-answer quality.


In [ ]:
from module_b._module_a import project as original
from module_b.evaluation import load_pipeline, evaluate_selected
DESIGN_PATH = PROJECT / "service/data/pipeline_design.json"
CASE_IDS = ["EVAL-FC-005", "EVAL-FC-003"]  # Independent task: select and justify your own pair.
pipeline = load_pipeline(DESIGN_PATH)
print("Available original case IDs:", [case["eval_id"] for case in pipeline.env["eval_cases"]])
for case_id in CASE_IDS:
    print("Original criterion:", original.eval_case_by_id(pipeline.env, case_id))
report = evaluate_selected(DESIGN_PATH, CASE_IDS)
for row in report["rows"]:
    print(row)
print("Reference-design hash:", report["design_revision"])
print("Deterministic reference evaluation; no claim about live wording or transport.")


### Independent investigation — 11 minutes

Write your safe signal and hypothesis in a text cell. Inspect the available original cases, choose a different relevant case and a useful companion, and justify both before editing `CASE_IDS`. Use `original.eval_case_by_id(pipeline.env, case_id)` to inspect a candidate. Preserve its input and criteria, then rerun `code-11`.

Record the actual result, original requirement, design hash and a bounded conclusion. Never replay a private log body into evaluation. If a case ID is unknown, choose from the printed original IDs; if the design file is missing, check the printed Live workspace and setup rather than inventing a replacement. An honest pass does not require changing a design.

| Safe signal / request ID | Hypothesis | Chosen cases and reasons | Original criteria | Actual result / design hash | Untested live claim |
|---|---|---|---|---|---|
| | | | | | |

**Peer check:** explain why the selected requirement answers your question and why the reference result does not certify every live answer.


<a id="ls12"></a>
## LS12 — Concept clinic

**Outcome:** Demo the streaming service with structured logging, redaction, and the eval feedback hook.

Explain the Colab concepts to a peer, using actual observations when available. Keep unavailable provider checks pending and label written or recorded examples. This is a formative concept clinic; no Campus project milestone is submitted.

| Minutes | Activity | What to explain |
|---|---|---|
| 00–12 | Orient and calibrate | Written `HTTP 200 → metadata → delta → error`; distinguish partial output from completion |
| 12–20 | Round 1: events and states | Stream/buffered observation, terminal state and exact response-to-log ID |
| 20–22 | Check | Can a reference-evaluator pass establish streamed wording quality? |
| 22–30 | Round 2: observation to test | Signal → hypothesis → original criterion → bounded result |
| 30–32 | Check | What did the local marker test establish, and what did saved-log inspection establish? |
| 32–40 | Round 3: useful safe logs | Exclusion, useful retained fields and limitations |
| 40–44 | Diagnose an identity mismatch | A different request's row cannot establish this request's outcome |
| 44–56 | Apply feedback in Colab | Choose one peer question; repeat or adjust the relevant check and improve your explanation |
| 56–60 | Connect and save | Explain a use in another application, then save notebook and ZIP |

In each round use 4 minutes to explain, 2 for peer questions, 1 for feedback and 1 to record an unresolved question. Rotate presenter, questioning peer and recorder. Explain all three ideas through the round's focus.

For the independent 12-minute application, name the peer question and relevant LS09–11 cell. Repeat a check or narrow an unsupported conclusion. Record what changed in your explanation and what is still unknown. Do not fill gaps with invented events or another invocation's log.

<a id="save-work"></a>
### Save your work after each session

Save your notebook with your predictions and explanations. Run `save-checkpoint`, then download the printed ZIP from the Files panel. Expected: source edits, model choices and selected safe observation summaries are preserved under `evidence/`. Each observation snapshot has a new timestamped filename, so exporting after restore preserves earlier snapshots even before you make a fresh request. The runtime JSONL file, raw response bodies, credentials and index are excluded. A saved summary is evidence from its original run, not proof that a restored runtime has executed it.


In [ ]:
if service is not None:
    service.stop()
(PROJECT / "evidence").mkdir(exist_ok=True)
(PROJECT / "evidence/live-models.json").write_text(json.dumps(
    {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}, indent=2) + "\n")
print("Save these configured model choices with your notebook:",
      {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")})
# Preserve only selected safe summaries; var/requests.jsonl is not exported.
from module_b.observability import safe_record
current_ids = {globals().get(name) for name in (
    "rejected_request_id", "clarification_request_id", "stream_request_id", "buffered_request_id")}
current_ids.discard(None)
log_path = PROJECT / "var" / "requests.jsonl"
saved_rows = []
if log_path.exists():
    for line in log_path.read_text().splitlines():
        row = json.loads(line)
        if row.get("request_id") in current_ids:
            saved_rows.append(safe_record(
                {**row, "_approved_models": (service_env["OPENROUTER_MODEL"], service_env["OPENROUTER_MODEL_V2"])},
                fields=tuple(row), routes=("/v1/diagnose", "/v1/diagnose-stream", "/v2/diagnose")))
summary = {
    "evidence_kind": "historical_live_notebook_observations",
    "source_revision": SOURCE_REVISION,
    "safe_request_records": saved_rows,
    "provider_comparison_attempted": bool(globals().get("RUN_PROVIDER", False)),
    "stream_event_types": [event.get("type") for event in globals().get("events", [])],
    "stream_request_id": globals().get("stream_request_id"),
    "buffered_request_id": globals().get("buffered_request_id"),
    "first_delta_seconds": globals().get("first_delta_seconds"),
    "stream_reader_seconds": globals().get("stream_elapsed_seconds"),
    "buffered_reader_seconds": globals().get("buffered_elapsed_seconds"),
    "stream_transport_error": globals().get("stream_transport_error"),
    "buffered_transport_error": globals().get("buffered_transport_error"),
    "reference_evaluation": globals().get("report"),
    "limit": "Original run only; no raw prompts, responses, headers or credentials. Reference results do not grade live wording."
}
OBSERVATION_OUT = PROJECT / "evidence" / f"live-session-observations-{time.time_ns()}.json"
OBSERVATION_OUT.write_text(json.dumps(summary, indent=2) + "\n")
print("Safe observation snapshot:", OBSERVATION_OUT.name)
CHECKPOINT_OUT = PROJECT.parent / f"sprint-3-live-{time.time_ns()}.zip"
print("Download:", export_live_workspace(PROJECT, CHECKPOINT_OUT))


## Watch out for

| Symptom | Check and recovery |
|---|---|
| Setup refuses an existing folder | Use a marked Live folder or a new destination; preserve unrelated work |
| Source edit has no effect | Save the file under the printed `PROJECT`, inspect it again, then run its start cell |
| `401`, `422` or `429` instead of stream events | Check the caller key, input and available allowance; these are response branches, not deltas |
| Provider comparison is pending | Keep it pending or explicitly prepare access/index and opt in; never substitute generated-looking output |
| Missing request ID or current log row | Attach observation once outside the guard, restart, make fresh requests and match exact UUIDs |
| Transport failure or no terminal event | Keep partial events and the error; completion is unproved |
| Restore has no live log or credentials | Inspect the saved historical evidence; start the restored source and make fresh requests |

## Quick reference

| Cell | Purpose |
|---|---|
| `ls09-source`, `code-04`, `ls09-boundaries` | Enable and inspect the stream route; test JSON branches |
| `prepare-provider`, `ls09-observed-start`, `code-05` | Explicit provider opt-in and client observation |
| `ls10-configure-source`, `ls10-configure-start`, `ls10-requests`, `code-07` | Edit logging, restart and correlate fresh records |
| `code-09` | Local minimization and separate saved-log inspection |
| `code-11` | Reuse the original Module A evaluator and criteria |
| `save-checkpoint` | Preserve edited source and selected safe summaries |

## Summary

You can explain incremental delivery and terminal state, retain useful operational fields without copying private inputs, and turn a log signal into a question for existing evaluation criteria. Keep your personal Colab explanations beside the observations that support them. Transfer these ideas to another application; keep any unexecuted check explicit.
